# Building a Drift Monitoring Dashboard

You've deployed a model. Now what?

This notebook shows how to set up continuous monitoring that answers:
1. Is my input data still similar to training data?
2. Is my model's behavior changing over time?
3. When should I retrain?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')
from drift_detection import KSTest, PSI

sns.set_style("whitegrid")
np.random.seed(42)

## Simulating Production Data

We'll simulate 12 weeks where:
- Weeks 1-4: Data matches training distribution
- Weeks 5-8: Gradual drift begins
- Weeks 9-12: Significant drift

In [ ]:
n_reference = 5000
reference_data = {
    'feature_1': np.random.normal(0.50, 0.10, n_reference),
    'feature_2': np.random.normal(100, 15, n_reference),
    'feature_3': np.random.normal(0.30, 0.08, n_reference)
}
reference_df = pd.DataFrame(reference_data)

print("Reference distribution statistics:")
reference_df.describe().round(3)

In [ ]:
def generate_weekly_data(week, samples_per_week=500):
    f1_mean, f1_std = 0.50, 0.10
    f2_mean, f2_std = 100, 15
    f3_mean, f3_std = 0.30, 0.08
    
    if week <= 4:
        pass  # No drift
    elif week <= 8:
        drift_factor = (week - 4) / 4
        f1_mean += 0.03 * drift_factor
        f2_mean += 5 * drift_factor
    else:
        f1_mean += 0.08
        f1_std *= 1.3
        f2_mean += 12
        f2_std *= 1.2
        f3_mean += 0.05
    
    return pd.DataFrame({
        'feature_1': np.random.normal(f1_mean, f1_std, samples_per_week),
        'feature_2': np.random.normal(f2_mean, f2_std, samples_per_week),
        'feature_3': np.random.normal(f3_mean, f3_std, samples_per_week),
        'week': week
    })

weekly_data = [generate_weekly_data(w) for w in range(1, 13)]
production_df = pd.concat(weekly_data, ignore_index=True)

print(f"Generated {len(production_df)} samples across 12 weeks")

## Running Weekly Drift Checks

In [ ]:
ks_detector = KSTest(alpha=0.05)
psi_calculator = PSI(n_bins=10)

monitoring_results = []
features = ['feature_1', 'feature_2', 'feature_3']

for week in range(1, 13):
    week_data = production_df[production_df['week'] == week]
    
    for feature in features:
        ref_values = reference_df[feature].values
        cur_values = week_data[feature].values
        
        ks_result = ks_detector.detect(ref_values, cur_values)
        psi_result = psi_calculator.calculate(ref_values, cur_values)
        
        monitoring_results.append({
            'week': week,
            'feature': feature,
            'ks_statistic': ks_result.statistic,
            'ks_pvalue': ks_result.p_value,
            'ks_drift': ks_result.drift_detected,
            'psi': psi_result.psi,
            'psi_level': psi_result.drift_level
        })

monitoring_df = pd.DataFrame(monitoring_results)
print(f"Collected {len(monitoring_df)} feature-week observations")

## The Dashboard View

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1 = axes[0]
for feature in features:
    feature_data = monitoring_df[monitoring_df['feature'] == feature]
    ax1.plot(feature_data['week'], feature_data['psi'], 'o-', label=feature, linewidth=2, markersize=8)

ax1.axhline(y=0.1, color='orange', linestyle='--', alpha=0.7, label='Warning (0.1)')
ax1.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Critical (0.2)')
ax1.axvspan(4.5, 8.5, alpha=0.1, color='orange')
ax1.axvspan(8.5, 12.5, alpha=0.1, color='red')

ax1.set_ylabel('PSI Score', fontsize=12)
ax1.set_title('Population Stability Index Over Time', fontsize=14)
ax1.legend(loc='upper left', ncol=2)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 0.35)

ax2 = axes[1]
for feature in features:
    feature_data = monitoring_df[monitoring_df['feature'] == feature]
    ax2.semilogy(feature_data['week'], feature_data['ks_pvalue'], 'o-', label=feature, linewidth=2, markersize=8)

ax2.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='alpha=0.05')
ax2.axvspan(4.5, 8.5, alpha=0.1, color='orange')
ax2.axvspan(8.5, 12.5, alpha=0.1, color='red')

ax2.set_xlabel('Week', fontsize=12)
ax2.set_ylabel('KS Test p-value (log)', fontsize=12)
ax2.set_title('KS Test Significance Over Time', fontsize=14)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

## Weekly Alert Report

In [ ]:
def generate_weekly_report(week, monitoring_df):
    week_data = monitoring_df[monitoring_df['week'] == week]
    
    print(f"\n{'='*60}")
    print(f" DRIFT MONITORING REPORT - WEEK {week}")
    print(f"{'='*60}")
    
    alerts = []
    warnings = []
    
    for _, row in week_data.iterrows():
        status = "STABLE"
        
        if row['psi'] > 0.2 or row['ks_drift']:
            status = "DRIFT"
            alerts.append(row['feature'])
        elif row['psi'] > 0.1:
            status = "WARNING"
            warnings.append(row['feature'])
        
        print(f"\n[{status}] {row['feature']}")
        print(f"   PSI: {row['psi']:.4f} ({row['psi_level']})")
        print(f"   KS p-value: {row['ks_pvalue']:.2e}")
    
    print(f"\n{'-'*60}")
    print(f"SUMMARY: {len(alerts)} alerts, {len(warnings)} warnings")
    
    if alerts:
        print(f"\nACTION REQUIRED: Significant drift detected.")

generate_weekly_report(3, monitoring_df)
generate_weekly_report(7, monitoring_df)
generate_weekly_report(11, monitoring_df)

## When to Retrain?

| Week | Status | Action |
|------|--------|--------|
| 1-4 | Stable | Continue monitoring |
| 5-6 | Warning | Investigate root cause |
| 7-8 | Mixed alerts | Prepare retraining pipeline |
| 9+ | Critical | **Retrain required** |

In [ ]:
print("\nDRIFT TIMELINE")
print("="*60)

for feature in features:
    feature_data = monitoring_df[monitoring_df['feature'] == feature]
    
    first_warning = feature_data[feature_data['psi'] > 0.1]['week'].min()
    first_critical = feature_data[feature_data['psi'] > 0.2]['week'].min()
    first_ks_drift = feature_data[feature_data['ks_drift']]['week'].min()
    
    print(f"\n{feature}:")
    print(f"  First PSI warning (>0.1):  Week {first_warning if pd.notna(first_warning) else 'Never'}")
    print(f"  First PSI critical (>0.2): Week {first_critical if pd.notna(first_critical) else 'Never'}")
    print(f"  First KS drift detected:   Week {first_ks_drift if pd.notna(first_ks_drift) else 'Never'}")

## Key Takeaways

1. **Monitor continuously** - Don't wait for complaints
2. **Use multiple metrics** - KS catches sudden shifts, PSI tracks gradual drift
3. **Set clear thresholds** - Know what triggers investigation vs. action
4. **Automate alerts** - This notebook becomes a scheduled job
5. **Document decisions** - When you don't retrain, document why